In [ ]:
#!pip install streamlit streamlit-drawable-canvas pillow google-generativeai

In [ ]:
import os

In [ ]:
import time

In [ ]:
import base64

In [ ]:
import json

In [ ]:
import re

In [ ]:
import streamlit as st

In [ ]:
from PIL import Image

In [ ]:
from io import BytesIO

In [ ]:
from streamlit_drawable_canvas import st_canvas

2026-02-10 22:18:58.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from getpass import getpass
Google_API_KEY = getpass("Enter your Google API key here :")

Enter your Google API key here :··········


**Configure Gemini**

In [ ]:
import google.generativeai as genai

genai.configure(api_key=Google_API_KEY)

VISION_MODEL = "gemini-pro-vision"
TEXT_MODEL = "gemini-pro"

In [ ]:
SYNONYMS = {
    "car": ["automobile", "vehicle"],
    "cat": ["kitty", "kitten"],
    "dog": ["puppy"],
    "cup": ["mug"],
    "house": ["home"],
    "tree": ["plant"],
    "phone": ["mobile"],
}

**Word Generator**

In [ ]:
def generate_words():
    model = genai.GenerativeModel(TEXT_MODEL)

    prompt = """
    Generate 10 very simple, single-word noun objects that are easy to draw in under 20 seconds.

    Rules:
    - Only common physical objects
    - No abstract concepts
    - No multi-word phrases
    - Output ONLY valid JSON
    - JSON format: {"words": ["cat", "house", ...]}
    """

    response = model.generate_content(
        prompt,
        generation_config={"temperature": 0.6}
    )

    try:
        data = json.loads(response.text)
        return data["words"]
    except json.decoder.JSONDecodeError:
        st.warning("Model did not return valid JSON for word generation. Please try again.")
        return ["cat", "dog", "house", "tree", "car", "cup", "phone", "book", "hat", "star"]

**Drawing Canvas**

In [ ]:
def draw_canvas():
  canvas = st_canvas(
      fill_color="rgba(255, 255, 255, 0)",
      stroke_width=5,
      stroke_color="#000000",
      background_color="#FFFFFF",
      height=300,
      width=300,
      drawing_mode="freedraw",
      key="canvas"
  )
  return canvas

**Convert Canvas to Base64**

In [ ]:
def image_to_base64(image):
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encoder(buffer.getvalue()).decode()

**Evaluation Logic**

In [ ]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", "", text)
    return text.strip()

In [ ]:
def evaluate(guess, target):
    return normalize(guess) == normalize(target)

**Gemini Guess**

In [ ]:
def gemini_guess(image_base64):
    model = genai.GenerativeModel(VISION_MODEL)

    prompt = """
    You are playing a drawing guessing game.

    Rules:
    - Guess ONLY ONE Word.
    - No Explanations
    - If unsure, return "Uncertain".
    - Output ONLY valid JSON.

    JSON format:
    {"guess": "<one_word_here>"}
    """

    image_part = {
        "mime_type": "image/png",
        "data": base64.b64decode(image_base64)
    }

    response = model.generate_content(
        [prompt, image_part],
        generation_config={"temperature": 0.2}
    )

    try:
        return json.loads(response.text)["guess"]
    except json.decoder.JSONDecodeError:
        st.warning("Model did not return valid JSON for guessing. Returning 'Uncertain'.")
        return "Uncertain"

**Streamlit Orchestration**

In [ ]:
st.title("🎨 Gemini QuickDraw Clone")

if "words" not in st.session_state:
    try:
        st.session_state.words = generate_words()
        st.session_state.target = st.session_state.words[0]
    except Exception as e:
        st.error(f"Failed to generate words: {e}")
        # Ensure target is still set to prevent AttributeError, using a default fallback
        st.session_state.words = ["default"] # Assign a default list to words
        st.session_state.target = st.session_state.words[0] # Assign target from the default list
        st.stop() # Attempt to stop further execution in this run

st.write(f"**Draw this:** `{st.session_state.target}`")

canvas = draw_canvas()

if st.button("Submit Drawing"):
    if canvas.image_data is not None:
        start = time.time()

        image = Image.fromarray(canvas.image_data.astype("uint8"))
        img_b64 = image_to_base64(image)

        guess = gemini_guess(img_b64)
        duration = round(time.time() - start, 2)

        correct = evaluate(guess, st.session_state.target)

        st.subheader(f"🤖 Gemini guessed: **{guess}**")
        st.write(f"⏱ Time taken: {duration}s")

        if correct:
            st.success("✅ Correct!")
        else:
            st.error("❌ Incorrect")

2026-02-10 22:19:59.905 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-10 22:19:59.977 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-10 22:19:59.977 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-10 22:19:59.979 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-10 22:19:59.979 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-10 22:19:59.980 Session state does not function when running a script without `streamlit run`
2026-02-10 22:20:01.072 404 POST /v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1083.13ms
2026-02-10 22:20:01.073 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-10 